# 05 · Sensitivity check: without 250 hPa

**Why:** 250 hPa is about 34,000 ft (FL340). Since the UA1093 incident, WindBorne balloons no longer hold neutral buoyancy between FL300 and FL400 (WindBorne, UA1093 update). Notebooks 02 to 04 use 250 hPa as one of three heights, so part of the result describes flying WindBorne now avoids.

**What changes here:** balloons only use **850 and 500 hPa**.
- **Passive:** the 850 and 500 hPa results from notebook 02 don't change. Only the 250 hPa column drops out.
- **Steered:** the pilots choose between two heights instead of three. Everything else matches notebook 04: both pilots, both tie-break rules, three gap definitions, 9 winters.

**The question:** does Iceland's position change when 250 hPa is removed?

In [1]:
import numpy as np
import pandas as pd
import balloon as b

SITES = {
    "Egilsstaðir":  (65.28, -14.40),
    "Keflavík":     (63.97, -22.60),
    "New York*":    (40.71, -74.01),
    "Cambridge MA": (42.36, -71.09),
    "Juneau":       (58.30, -134.42),
}
order = list(SITES)
WINTERS = range(2013, 2022)
USE = [0, 1]                       # indices into b.LEVELS = [850, 500, 250]: 850 and 500 only

stations = b.active_stations()
dist_km = b.nearest_station_km_grid(stations)
gap_masks = {km: dist_km > km for km in [300, 500, 800]}
region_lats, _ = b.region_grid()
arctic_500 = gap_masks[500] & (region_lats[:, None] >= 60)

## 1. Steered flights with two heights (about 3 minutes)

Balloons start at 500 hPa, as in notebook 04.

In [2]:
def hours_in(lats, lons, mask, H):
    return b.in_gap(lats[1:H + 1], lons[1:H + 1], mask).sum(axis=0)

rows = []
for winter in WINTERS:
    w = b.load_winter(winter)
    winds = [b.make_wind(w["u"][:, i], w["v"][:, i], w["time"]) for i in USE]
    days = pd.date_range(f"{winter}-12-01", f"{winter + 1}-03-01", freq="D", inclusive="left")
    site_idx = np.tile(np.arange(len(order)), len(days))
    day_idx = np.repeat(np.arange(len(days)), len(order))
    lat0 = np.array([SITES[n][0] for n in order])[site_idx]
    lon0 = np.array([SITES[n][1] for n in order])[site_idx]
    t0 = (days.values[day_idx] - w["time"][0]) / np.timedelta64(1, "s")
    for lookahead in ["persistence", "perfect"]:
        for tiebreak in ["stay", "closest"]:
            for km, mask in gap_masks.items():
                lats, lons, _ = b.fly_steered(winds, lat0, lon0, t0, 21 * 24, mask, tiebreak=tiebreak,
                                              lookahead=lookahead, start_level=1)   # 1 = 500 hPa here
                rec = dict(winter=winter, site=np.array(order)[site_idx], lookahead=lookahead,
                           tiebreak=tiebreak, gap_km=km,
                           gap_7d=hours_in(lats, lons, mask, 168), gap_21d=hours_in(lats, lons, mask, 504))
                if km == 500:
                    rec["arctic_21d"] = hours_in(lats, lons, arctic_500, 504)
                rows.append(pd.DataFrame(rec))
    print("winter", winter, "done")
two = pd.concat(rows, ignore_index=True)
two.to_csv("steered_no250_runs.csv", index=False)

winter 2013 done
winter 2014 done
winter 2015 done
winter 2016 done
winter 2017 done
winter 2018 done
winter 2019 done
winter 2020 done
winter 2021 done


## 2. Side by side: three heights (notebook 04) vs two heights

Median gap-hours per launch over 21 days, 500 km gap.

In [3]:
three = pd.read_csv("steered_runs.csv")
def medians(df, km=500, col="gap_21d"):
    return df[df.gap_km == km].pivot_table(index="site", columns=["lookahead", "tiebreak"],
                                           values=col, aggfunc="median").reindex(order)
comparison = pd.concat({"3 heights": medians(three), "2 heights (no 250)": medians(two)}, axis=1)
comparison

3 heights                         2 heights (no 250)        \
lookahead      perfect       persistence                  perfect         
tiebreak       closest  stay     closest  stay            closest  stay   
site                                                                      
Egilsstaðir       47.0  22.0        43.0  20.5               40.5  21.0   
Keflavík          55.0  27.0        53.0  27.0               44.5  23.0   
New York*         59.0  28.0        57.0  27.0               54.0  27.0   
Cambridge MA      62.0  34.0        56.0  31.0               53.0  33.0   
Juneau            71.0  35.5        70.0  35.0               54.0  32.0   

                                
lookahead    persistence        
tiebreak         closest  stay  
site                            
Egilsstaðir         35.0  20.0  
Keflavík            40.0  23.5  
New York*           48.0  26.0  
Cambridge MA        53.0  31.0  
Juneau              53.0  32.0

## 3. Does the ranking survive? (Ranks: 1 = most gap-hours, 21 days)

In [4]:
r3 = three.pivot_table(index="site", columns=["lookahead", "tiebreak", "gap_km"], values="gap_21d",
                       aggfunc="median").reindex(order).rank(ascending=False, method="min").astype(int)
r2 = two.pivot_table(index="site", columns=["lookahead", "tiebreak", "gap_km"], values="gap_21d",
                     aggfunc="median").reindex(order).rank(ascending=False, method="min").astype(int)
print("Three heights:"); display(r3)
print("Two heights (no 250 hPa):"); display(r2)
print("Rankings identical in", (r3 == r2).all().sum(), "of", r3.shape[1], "columns")

Three heights:


lookahead    perfect                      persistence                     
tiebreak     closest         stay             closest         stay        
gap_km           300 500 800  300 500 800         300 500 800  300 500 800
site                                                                      
Egilsstaðir        3   5   5    1   5   4           5   5   5    2   5   4
Keflavík           2   4   4    2   4   4           2   4   4    1   3   4
New York*          5   3   2    5   3   2           3   2   2    5   3   3
Cambridge MA       4   2   1    4   2   1           3   3   1    4   2   1
Juneau             1   1   3    3   1   2           1   1   3    3   1   2

Two heights (no 250 hPa):


lookahead    perfect                      persistence                     
tiebreak     closest         stay             closest         stay        
gap_km           300 500 800  300 500 800         300 500 800  300 500 800
site                                                                      
Egilsstaðir        5   5   5    4   5   4           5   5   5    4   5   4
Keflavík           3   4   4    2   4   4           2   4   4    1   4   4
New York*          2   1   1    5   3   2           4   3   2    5   3   2
Cambridge MA       1   3   2    3   1   1           3   1   1    3   2   1
Juneau             4   1   3    1   2   2           1   1   3    2   1   3

Rankings identical in 2 of 12 columns


## 4. The Arctic part, with and without 250 hPa (mean hours over 21 days, 500 km gap)

In [5]:
def arctic(df):
    s = df[df.gap_km == 500]
    return s.pivot_table(index="site", columns=["lookahead", "tiebreak"], values="arctic_21d",
                         aggfunc="mean").reindex(order)
pd.concat({"3 heights": arctic(three), "2 heights (no 250)": arctic(two)}, axis=1).round(1)

3 heights                         2 heights (no 250)        \
lookahead      perfect       persistence                  perfect         
tiebreak       closest  stay     closest  stay            closest  stay   
site                                                                      
Egilsstaðir       49.4  19.3        43.0  18.3               35.2  16.2   
Keflavík          48.9  21.9        46.4  20.5               34.2  18.6   
New York*         34.9  11.8        33.2  10.5               24.2  10.4   
Cambridge MA      35.8  14.8        32.5  11.2               25.3  12.2   
Juneau            50.0  16.7        45.9  16.0               29.8  13.5   

                                
lookahead    persistence        
tiebreak         closest  stay  
site                            
Egilsstaðir         29.5  15.6  
Keflavík            31.9  17.8  
New York*           22.9  10.0  
Cambridge MA        24.6  11.6  
Juneau              30.7  12.6

## Passive results without 250 hPa

No new runs are needed: these are notebook 02's 850 and 500 hPa rows (median gap-hours per launch).

In [6]:
passive = pd.read_csv("passive_runs.csv")
passive[passive.level.isin([850, 500])].pivot_table(
    index="site", columns="level", values=["gap_h_500km_7d", "gap_h_500km_21d"], aggfunc="median").reindex(order)

gap_h_500km_21d       gap_h_500km_7d      
level                    500   850            500   850
site                                                   
Egilsstaðir             12.0  19.0            1.0   7.0
Keflavík                14.0  27.0            2.0  13.0
New York*               18.0  38.0           14.0  25.0
Cambridge MA            20.0  37.0           16.0  24.5
Juneau                  21.0   0.0           12.0   0.0

## What to read off this (Senan writes the findings)

- If the rankings are the same, the main result holds without the height WindBorne avoids. That's one sentence in the write-up.
- If they change, say how, and which version better matches how WindBorne flies.
- **Remaining limit:** we don't know WindBorne's actual altitude range, including whether GSBs float as low as 850 hPa (about 5,000 ft). Their flight logs would settle that.